# Manual CSV Load into MongoDB


In [42]:
import pandas as pd
from pymongo import MongoClient
from pymongo.server_api import ServerApi
import streamlit as st
from hampel import hampel


Load Data

In [43]:
data = pd.read_csv("./.streamlit/25-01-27 Bienenwaage1.csv")
print(data.dtypes)
data

created_at     object
entry_id        int64
field1         object
field2        float64
field3         object
latitude      float64
longitude     float64
elevation     float64
status        float64
dtype: object


C:\Users\timwy\AppData\Local\Temp\ipykernel_17604\4104083099.py:1: DtypeWarning: Columns (2,4) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv("./.streamlit/25-01-27 Bienenwaage1.csv")


,created_at,entry_id,field1,field2,field3,latitude,longitude,elevation,status
0,2020-10-14T17:03:37+02:00,1,49.63,78.0,13.1,NaN,NaN,NaN,NaN
1,2020-10-14T17:13:41+02:00,2,49.77,95.0,20.3,NaN,NaN,NaN,NaN
2,2020-10-14T17:23:45+02:00,3,49.79,95.0,23.1,NaN,NaN,NaN,NaN
3,2020-10-14T17:33:49+02:00,4,49.78,86.0,23.4,NaN,NaN,NaN,NaN
4,2020-10-14T17:43:53+02:00,5,49.77,79.0,23.5,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
189685,2025-01-27T17:46:54+01:00,189686,40.64,88.0,10.20,NaN,NaN,NaN,NaN
189686,2025-01-27T17:56:55+01:00,189687,40.63,88.0,10.10,NaN,NaN,NaN,NaN
189687,2025-01-27T18:06:57+01:00,189688,40.63,88.0,10.20,NaN,NaN,NaN,NaN
189688,2025-01-27T18:16:58+01:00,189689,40.63,88.0,10.20,NaN,NaN,NaN,NaN


In [44]:
data.rename(columns={'field4': 'timestamp', 'field1': 'weight', 'field2': 'humidity', 'field3': 'temperature', 'created_at':'timestamp'}, inplace=True)
print(data.dtypes)
data = data[['timestamp', 'weight', 'humidity', 'temperature']]
# data['timestamp'].fillna(data['created_at'], inplace=True)
data.head()
data.tail()

df = data[['timestamp', 'weight', 'humidity', 'temperature']]


display(df)

timestamp       object
entry_id         int64
weight          object
humidity       float64
temperature     object
latitude       float64
longitude      float64
elevation      float64
status         float64
dtype: object


,timestamp,weight,humidity,temperature
0,2020-10-14T17:03:37+02:00,49.63,78.0,13.1
1,2020-10-14T17:13:41+02:00,49.77,95.0,20.3
2,2020-10-14T17:23:45+02:00,49.79,95.0,23.1
3,2020-10-14T17:33:49+02:00,49.78,86.0,23.4
4,2020-10-14T17:43:53+02:00,49.77,79.0,23.5
...,...,...,...,...
189685,2025-01-27T17:46:54+01:00,40.64,88.0,10.20
189686,2025-01-27T17:56:55+01:00,40.63,88.0,10.10
189687,2025-01-27T18:06:57+01:00,40.63,88.0,10.20
189688,2025-01-27T18:16:58+01:00,40.63,88.0,10.20


## cleaning

In [49]:
data['timestamp'] = pd.to_datetime(data['timestamp'], format='mixed', yearfirst=True, utc=True)
print(data[data['timestamp'].isna()])
# Remove rows where 'weight' is equal to "~"
data = data[data['weight'] != "~"]
# Convert 'weight' column to float
data.loc[:, 'weight'] = data['weight'].astype(float)
data = data[data['weight'] >= 0]

Empty DataFrame
Columns: [timestamp, weight, humidity, temperature]
Index: []


C:\Users\timwy\AppData\Local\Temp\ipykernel_17604\663850876.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['timestamp'] = pd.to_datetime(data['timestamp'], format='mixed', yearfirst=True, utc=True)


In [50]:
data['temperature'] = pd.to_numeric(data['temperature'], errors='coerce')
display(data[data['temperature'].isna()])

,timestamp,weight,humidity,temperature
5118,2020-11-21 06:48:53+00:00,0.36,NaN,NaN
49848,2021-10-18 21:04:51+00:00,35.79,NaN,NaN
76539,2022-05-19 16:41:10+00:00,107.78,NaN,NaN
113969,2023-05-20 20:23:20+00:00,24.59,NaN,NaN
113970,2023-05-20 20:33:24+00:00,24.59,NaN,NaN
...,...,...,...,...
161585,2024-06-28 14:37:48+00:00,43.12,NaN,NaN
161586,2024-06-28 14:47:49+00:00,43.11,NaN,NaN
161587,2024-06-28 14:57:51+00:00,43.1,NaN,NaN
161588,2024-06-28 15:07:52+00:00,43.08,NaN,NaN


In [51]:

data['humidity'] = data['humidity'].astype(float)
data['temperature'] = data['temperature'].astype(float)


In [52]:
data.set_index('timestamp', inplace=True)
# Apply the Hampel filter
WINDOW_SIZE = 144 # 10 min/datapoint | 6 datapoint/hour | 144 datapoints/day
result = hampel(data['weight'], window_size=WINDOW_SIZE, n_sigma=3.0)

data_weight_filtered = result.filtered_data
outlier_indices = result.outlier_indices
medians = result.medians
mad_values = result.median_absolute_deviations
thresholds = result.thresholds
print(data_weight_filtered)

0         49.630001
1         49.770000
2         49.790001
3         49.779999
4         49.770000
            ...    
187694    40.639999
187695    40.630001
187696    40.630001
187697    40.630001
187698    40.639999
Length: 187699, dtype: float32


In [53]:
data_weight_filtered.index = data.index
data['weight'] = data_weight_filtered

data = data.reset_index()
#create a column beehive_id with the value 2
data['beehive_id'] = "1"
data['timestamp'] = data['timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S')
data


,timestamp,weight,humidity,temperature,beehive_id
0,2020-10-14 15:03:37,49.630001,78.0,13.1,1
1,2020-10-14 15:13:41,49.770000,95.0,20.3,1
2,2020-10-14 15:23:45,49.790001,95.0,23.1,1
3,2020-10-14 15:33:49,49.779999,86.0,23.4,1
4,2020-10-14 15:43:53,49.770000,79.0,23.5,1
...,...,...,...,...,...
187694,2025-01-27 16:46:54,40.639999,88.0,10.2,1
187695,2025-01-27 16:56:55,40.630001,88.0,10.1,1
187696,2025-01-27 17:06:57,40.630001,88.0,10.2,1
187697,2025-01-27 17:16:58,40.630001,88.0,10.2,1


In [54]:
uri = st.secrets["mongodb"]["uri"]
# Create a new client and connect to the server
client = MongoClient(uri, server_api=ServerApi('1'))
# MongoDB Atlas Connection
db = client["beehive_monitoring"]  # Database name
collection = db["bee_sensor_telemetry"]  # Collection name

def insert_data_to_mongodb(beehive_id, weight, temperature, humidity, timestamp):
    """
    Inserts the received data into MongoDB Atlas.
    """
    data = {
        "beehive_id": beehive_id,
        "weight": float(weight),
        "temperature": float(temperature) if temperature != "nan" else None,
        "humidity": float(humidity) if humidity != "nan" else None,
        "timestamp": timestamp
    }
    
    try:
        result = collection.insert_one(data)
        print(f"Data inserted into MongoDB with ID: {result.inserted_id}")
    except Exception as e:
        print(f"Error inserting data to MongoDB: {e}")

In [27]:
# Example data for testing
beehive_id = "hive_01"
weight = 12.5
temperature = 35.2
humidity = 60.5
timestamp = "2023-10-01T12:00:00Z"

# Insert the example data into MongoDB
# insert_data_to_mongodb(beehive_id, weight, temperature, humidity, timestamp)

In [28]:
records = df.to_dict(orient='records')
display(records)

[{'timestamp': '2020-10-08T17:47:31+02:00',
  'weight': 46.38,
  'humidity': 93.0,
  'temperature': 19.8,
  'beehive_id': '2'},
 {'timestamp': '2020-10-08T17:57:35+02:00',
  'weight': 46.43,
  'humidity': 93.0,
  'temperature': 19.8,
  'beehive_id': '2'},
 {'timestamp': '2020-10-08T18:07:40+02:00',
  'weight': 46.4,
  'humidity': 93.0,
  'temperature': 19.8,
  'beehive_id': '2'},
 {'timestamp': '2020-10-08T18:17:44+02:00',
  'weight': 46.42,
  'humidity': 93.0,
  'temperature': 19.8,
  'beehive_id': '2'},
 {'timestamp': '2020-10-08T18:27:48+02:00',
  'weight': 46.36,
  'humidity': 93.0,
  'temperature': 19.9,
  'beehive_id': '2'},
 {'timestamp': '2020-10-08T18:37:53+02:00',
  'weight': 46.37,
  'humidity': 93.0,
  'temperature': 20.0,
  'beehive_id': '2'},
 {'timestamp': '2020-10-08T18:47:57+02:00',
  'weight': 46.34,
  'humidity': 93.0,
  'temperature': 19.9,
  'beehive_id': '2'},
 {'timestamp': '2020-10-08T18:58:01+02:00',
  'weight': 46.36,
  'humidity': 93.0,
  'temperature': 19.9,

In [56]:
def load_dataframe_to_mongodb(df, collection):
    """
    Loads data from a pandas DataFrame into a MongoDB collection.
    
    Parameters:
    df (pandas.DataFrame): The DataFrame containing the data to be loaded.
    collection (pymongo.collection.Collection): The MongoDB collection where the data will be inserted.
    """
    records = df.to_dict(orient='records')
    try:
        result = collection.insert_many(records)
        print(f"Inserted {len(result.inserted_ids)} records into MongoDB.")
    except Exception as e:
        print(f"Error inserting data to MongoDB: {e}")

# Load the data from the DataFrame into MongoDB
load_dataframe_to_mongodb(data, collection)

Inserted 187699 records into MongoDB.


In [55]:
result = collection.delete_many({"beehive_id": "1"})
print(f"Deleted {result.deleted_count} documents.")


Deleted 183075 documents.
